In [1]:
import requests
import time
from tqdm import tqdm
from urllib.parse import unquote
import pandas as pd
from bs4 import BeautifulSoup
import re

# =========================
# LOAD DATA
# =========================

df = pd.read_csv("politicians_network_nodes.csv")

API_URL = "https://en.wikipedia.org/w/api.php"

session = requests.Session()
session.trust_env = False
session.headers.update({
    "User-Agent": "PoliticiansNetwork/1.0 (s245290@dtu.dk)"
})


# =========================
# HELPERS
# =========================

def title_from_url(url: str) -> str:
    if not isinstance(url, str) or "/wiki/" not in url:
        return ""
    return unquote(url.rstrip("/").split("/wiki/")[-1])


def fetch_intro(title: str) -> str:
    params = {
        "action": "query",
        "format": "json",
        "titles": title,
        "prop": "extracts",
        "exintro": True,
        "explaintext": True,
        "redirects": 1,
    }

    r = session.get(API_URL, params=params, timeout=20)
    r.raise_for_status()

    pages = r.json().get("query", {}).get("pages", {})
    if not pages:
        return ""

    page = next(iter(pages.values()))
    return page.get("extract", "") or ""


def fetch_sections(title: str):
    params = {
        "action": "parse",
        "format": "json",
        "page": title,
        "prop": "sections",
        "redirects": 1,
    }

    r = session.get(API_URL, params=params, timeout=20)
    r.raise_for_status()

    return r.json().get("parse", {}).get("sections", [])


def extract_first_paragraphs_from_html(html: str, max_paragraphs=2) -> str:
    soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["sup", "style", "script", "table", "ul", "ol"]):
        tag.decompose()

    paragraphs = []
    for p in soup.find_all("p"):
        text = p.get_text(separator=" ")
        text = re.sub(r"\s+", " ", text).strip()

        if len(text) > 80:
            paragraphs.append(text)

        if len(paragraphs) >= max_paragraphs:
            break

    return "\n\n".join(paragraphs)


def section_type(section_title: str):
    title = section_title.lower()

    skip_patterns = [
        "early life",
        "education",
        "personal life",
        "death",
        "legacy",
        "honours",
        "honors",
        "awards",
        "electoral history",
        "bibliography",
        "references",
        "external links",
        "see also",
        "notes",
    ]

    positions_patterns = [
        "political positions",
        "political views",
        "views",
        "ideology",
        "policies",
        "foreign policy",
        "domestic policy",
    ]

    career_patterns = [
        "political career",
        "career",
        "presidency",
        "prime minister",
        "government",
        "minister",
        "parliament",
        "senate",
        "leadership",
    ]

    if any(skip in title for skip in skip_patterns):
        return None

    if any(pattern in title for pattern in positions_patterns):
        return "positions"

    if any(pattern in title for pattern in career_patterns):
        return "career"

    return None


def fetch_section_paragraphs(title: str, section_index: str, max_paragraphs=2) -> str:
    params = {
        "action": "parse",
        "format": "json",
        "page": title,
        "prop": "text",
        "section": section_index,
        "redirects": 1,
    }

    r = session.get(API_URL, params=params, timeout=20)
    r.raise_for_status()

    html = r.json().get("parse", {}).get("text", {}).get("*", "")
    return extract_first_paragraphs_from_html(html, max_paragraphs=max_paragraphs)


def fetch_relevant_wikipedia_text(url: str) -> dict:
    title = title_from_url(url)

    if not title:
        return {
            "intro_text": "",
            "career_text": "",
            "positions_text": "",
            "article_text": "",
            "section_titles": []
        }

    intro = fetch_intro(title)
    sections = fetch_sections(title)

    career_text = ""
    positions_text = ""
    selected_titles = []

    career_sections_used = 0
    positions_sections_used = 0

    for sec in sections:
        sec_title = sec.get("line", "")
        sec_index = sec.get("index", "")
        sec_type = section_type(sec_title)

        if sec_type == "career" and career_sections_used < 1:
            text = fetch_section_paragraphs(
                title,
                sec_index,
                max_paragraphs=2
            )

            if text:
                career_text = text
                selected_titles.append(sec_title)
                career_sections_used += 1

        elif sec_type == "positions" and positions_sections_used < 1:
            text = fetch_section_paragraphs(
                title,
                sec_index,
                max_paragraphs=1
            )

            if text:
                positions_text = text
                selected_titles.append(sec_title)
                positions_sections_used += 1

    article_parts = [intro, career_text, positions_text]
    article_text = "\n\n".join([part for part in article_parts if part])

    return {
        "intro_text": intro,
        "career_text": career_text,
        "positions_text": positions_text,
        "article_text": article_text,
        "section_titles": selected_titles
    }


# =========================
# BUILD CORPUS
# =========================

def build_text_corpus(df, max_articles=None, delay=0.3):
    rows = []

    subset = df[df["wikipedia_url"].notna() & (df["wikipedia_url"] != "")].copy()

    if max_articles is not None:
        subset = subset.head(max_articles)

    for _, row in tqdm(subset.iterrows(), total=len(subset), desc="Fetching selected article text"):
        try:
            result = fetch_relevant_wikipedia_text(row["wikipedia_url"])

            rows.append({
                "wikidata": row["wikidata"],
                "intro_text": result["intro_text"],
                "career_text": result["career_text"],
                "positions_text": result["positions_text"],
                "article_text": result["article_text"],
                "section_titles": result["section_titles"],
            })

        except Exception as e:
            print(f"Error fetching {row['wikipedia_url']}: {e}")

            rows.append({
                "wikidata": row["wikidata"],
                "intro_text": "",
                "career_text": "",
                "positions_text": "",
                "article_text": "",
                "section_titles": [],
            })

        time.sleep(delay)

    return pd.DataFrame(rows)


# =========================
# RUN
# =========================

text_df = build_text_corpus(df, max_articles=None, delay=0.3)

df = df.merge(text_df, on="wikidata", how="left")

for col in ["intro_text", "career_text", "positions_text", "article_text"]:
    df[col] = df[col].fillna("")

output_csv = "politicians_network_nodes_with_selected_text.csv"
df.to_csv(output_csv, index=False)

print(f"Saved updated dataframe to: {output_csv}")
print("Non-empty intro:", (df["intro_text"].str.len() > 0).sum())
print("Non-empty career:", (df["career_text"].str.len() > 0).sum())
print("Non-empty positions:", (df["positions_text"].str.len() > 0).sum())
print("Average text length in words:", df["article_text"].str.split().str.len().mean())

print(df[["name", "wikipedia_url", "section_titles", "article_text"]].head(3).to_string())

if "G_lcc" in globals():
    for _, row in df.iterrows():
        qid = row["wikidata"]
        if qid in G_lcc:
            G_lcc.nodes[qid]["article_text"] = row["article_text"]
            G_lcc.nodes[qid]["intro_text"] = row["intro_text"]
            G_lcc.nodes[qid]["career_text"] = row["career_text"]
            G_lcc.nodes[qid]["positions_text"] = row["positions_text"]

    print("\nSelected text stored on network nodes.")

print("Ready for TF-IDF, topic modelling, NER, embeddings, etc.")

Fetching selected article text:  32%|███▏      | 1331/4117 [28:18<52:53,  1.14s/it]  

Error fetching https://en.wikipedia.org/wiki/Steve_Knight_(politician): ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))


Fetching selected article text:  33%|███▎      | 1357/4117 [30:59<49:52,  1.08s/it]   

Error fetching https://en.wikipedia.org/wiki/Buddy_Carter: HTTPSConnectionPool(host='en.wikipedia.org', port=443): Read timed out. (read timeout=20)


Fetching selected article text:  33%|███▎      | 1358/4117 [32:46<25:05:55, 32.75s/it]

...

Error fetching https://en.wikipedia.org/wiki/Donald_J._Irwin: HTTPSConnectionPool(host='en.wikipedia.org', port=443): Max retries exceeded with url: /w/api.php?action=query&format=json&titles=Donald_J._Irwin&prop=extracts&exintro=True&explaintext=True&redirects=1 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x12901cc50>: Failed to resolve 'en.wikipedia.org' ([Errno 8] nodename nor servname provided, or not known)"))


Fetching selected article text:  36%|███▌      | 1472/4117 [45:37<14:37,  3.01it/s]

Error fetching https://en.wikipedia.org/wiki/Jerry_Dyer: HTTPSConnectionPool(host='en.wikipedia.org', port=443): Max retries exceeded with url: /w/api.php?action=query&format=json&titles=Jerry_Dyer&prop=extracts&exintro=True&explaintext=True&redirects=1 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x12917f610>: Failed to resolve 'en.wikipedia.org' ([Errno 8] nodename nor servname provided, or not known)"))


Fetching selected article text:  39%|███▉      | 1606/4117 [50:22<1:18:45,  1.88s/it] 


KeyboardInterrupt: 